In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
# Функции активации и их производные
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    return x * (1 - x)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

In [ ]:
# Класс единичного нейрона
class FormalNeuron:
    def __init__(self, n_inputs, activation='sigmoid'):
        self.weights = np.random.randn(n_inputs) * 0.01
        self.bias = np.random.randn() * 0.01
        self.activation = activation
        self.cache = {}

    def forward(self, x):
        self.cache['x'] = x
        z = np.dot(x, self.weights) + self.bias
        self.cache['z'] = z
        
        if self.activation == 'sigmoid':
            a = sigmoid(z)
        elif self.activation == 'relu':
            a = relu(z)
        else:
            a = z
        
        self.cache['a'] = a
        return a
    
    def backward(self, dz):
        # dz shape: (batch_size,) -> reshape to (batch_size, 1) for matrix operations
        if dz.ndim == 1:
            dz = dz.reshape(-1, 1)

        dw = np.dot(self.cache['x'].T, dz).flatten()
        db = np.sum(dz)
        dx = np.dot(dz, self.weights.reshape(1, -1))
        return dw, db, dx

    def update(self, dw, db, learning_rate):
        self.weights -= learning_rate * dw
        self.bias -= learning_rate * db

In [ ]:
# Класс многослойной нейросети
class MultiLayerNetwork:
    def __init__(self, layer_sizes, activations):
        """
        layer_sizes: список с количеством нейронов в каждом слое
        activations: список функций активации для каждого слоя
        """
        self.layers = []
        self.activations = activations
        
        for i in range(len(layer_sizes) - 1):
            layer = []
            for _ in range(layer_sizes[i + 1]):
                neuron = FormalNeuron(layer_sizes[i], activation=activations[i])
                layer.append(neuron)
            self.layers.append(layer)
        
        self.cache = []

    def forward(self, X):
        """
        X: массив входных данных (batch_size, n_features)
        """
        self.cache = []
        activations = [X]
        
        for layer_idx, layer in enumerate(self.layers):
            layer_output = []
            for neuron in layer:
                output = neuron.forward(activations[-1])
                layer_output.append(output)
            
            layer_output = np.column_stack(layer_output)
            activations.append(layer_output)
            self.cache.append([neuron for neuron in layer])
        
        return activations[-1]

    def backward(self, X, y_true, learning_rate):
        batch_size = X.shape[0]
        output_pred = self.forward(X)
        
        # Compute initial gradient at output
        error = (output_pred - y_true)  # shape: (batch_size, n_output_neurons)
        
        # Backpropagate through layers
        for layer_idx in range(len(self.layers) - 1, -1, -1):
            layer = self.layers[layer_idx]
            new_error = np.zeros((batch_size, len(self.layers[layer_idx - 1]) if layer_idx > 0 else X.shape[1]))
            
            for neuron_idx, neuron in enumerate(layer):
                # Get derivative of activation function
                if neuron.activation == 'sigmoid':
                    deriv = sigmoid_derivative(neuron.cache['a'])
                elif neuron.activation == 'relu':
                    deriv = relu_derivative(neuron.cache['z'])
                else:
                    deriv = np.ones_like(neuron.cache['a'])
                
                # Extract error for this neuron and multiply by activation derivative
                dz = error[:, neuron_idx] * deriv  # shape: (batch_size,)
                
                dw, db, dx = neuron.backward(dz)
                neuron.update(dw / batch_size, db / batch_size, learning_rate)
                
                # Accumulate error for next layer
                new_error += dx
            
            error = new_error

    def train(self, X_train, y_train, epochs, learning_rate, batch_size=32):
        losses = []
        
        for epoch in range(epochs):
            # Shuffle data
            indices = np.random.permutation(len(X_train))
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]
            
            epoch_loss = 0
            
            # Mini-batch training
            for i in range(0, len(X_train), batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.backward(X_batch, y_batch, learning_rate)
                
                pred = self.forward(X_batch)
                loss = np.mean((pred - y_batch) ** 2)
                epoch_loss += loss
            
            avg_loss = epoch_loss / (len(X_train) // batch_size)
            losses.append(avg_loss)
            
            if (epoch + 1) % 50 == 0:
                print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")
        
        return losses

    def predict(self, X):
        return self.forward(X)

In [ ]:
# Загружаем Wine Quality датасет
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
data = pd.read_csv(url, sep=';')

# Проверяем данные
print(f"Dataset shape: {data.shape}")
print(f"Quality classes: {sorted(data['quality'].unique())}")
print(data.head())

In [ ]:
# Преобработка данных
# Классификация качества: низкое (3-4), среднее (5-6), высокое (7-8)
def map_quality_to_class(quality):
    if quality <= 4:
        return 0
    elif quality <= 6:
        return 1
    else:
        return 2

data['quality_class'] = data['quality'].apply(map_quality_to_class)

# Берём только признаки (без оригинального quality)
X = data.drop(['quality', 'quality_class'], axis=1).values
y = data['quality_class'].values.reshape(-1, 1)

# One-hot encoding для целевой переменной
y_onehot = np.zeros((y.shape[0], 3))
for i in range(y.shape[0]):
    y_onehot[i, int(y[i, 0])] = 1

print(f"Features: {X.shape[1]}")
print(f"Classes: 3 (low, medium, high)")
print(f"Class distribution: {np.bincount(y.flatten())}")

In [ ]:
# Разделение данных и нормализация
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Создаём и обучаем сеть
# Архитектура: 11 -> 16 -> 8 -> 3
np.random.seed(42)

network = MultiLayerNetwork(
    layer_sizes=[11, 16, 8, 3],
    activations=['relu', 'relu', 'sigmoid']
)

# Обучение
print("Обучение сети...")
losses = network.train(X_train, y_train, epochs=500, learning_rate=0.01, batch_size=32)
print(f"\nФинальная потеря: {losses[-1]:.4f}")

In [ ]:
# График потерь
plt.figure(figsize=(10, 6))
plt.plot(losses, linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training Loss over Epochs', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Оценка на тестовом наборе
y_pred_train = network.predict(X_train)
y_pred_test = network.predict(X_test)

# Получаем классы (индекс максимального значения)
pred_train_class = np.argmax(y_pred_train, axis=1)
pred_test_class = np.argmax(y_pred_test, axis=1)

true_train_class = np.argmax(y_train, axis=1)
true_test_class = np.argmax(y_test, axis=1)

# Вычисляем точность
train_accuracy = np.mean(pred_train_class == true_train_class)
test_accuracy = np.mean(pred_test_class == true_test_class)

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Матрица ошибок
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(true_test_class, pred_test_class)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(true_test_class, pred_test_class, 
                          target_names=['Low', 'Medium', 'High']))